# Producer Pattern 1 of 5 — Fire-and-Forget
### A zero-to-production tutorial for junior Python/Kafka engineers

> **Guiding idea:** Fire-and-forget means your application **does not track the final result of each individual message**. It does **not** mean Kafka skips acknowledgements, and it does **not** mean you can ignore queue pressure or shutdown.

We will build a realistic **non-critical website analytics** publisher (`page_view` events). Losing an occasional analytics event may be tolerable; losing a payment or an order is not.

**After this notebook, you should be able to:**
1. Describe who actually sends a Kafka message.
2. Explain `produce()`, producer `poll()`, and `flush()` without memorizing internals.
3. Publish structured JSON events asynchronously.
4. Recognize local queue pressure and handle `BufferError` with a deadline.
5. Explain why `flush() == 0` is *not* a successful-delivery certificate.
6. Implement and test a small reusable fire-and-forget producer.
7. Decide when to use callback-confirmed publishing instead.

**Learning format:** Each section has an **objective → concept → small runnable cell → expected observation → engineering takeaway**. The advanced experiments are explicitly opt-in.

## Learning roadmap

| Section | Objective | Level |
|---|---|---|
| 0 | Prepare a safe local Kafka lab | Setup |
| 1 | Understand fire-and-forget with one mental model | Beginner |
| 2 | Publish one real JSON event | Beginner |
| 3 | Understand `produce()`, `poll()`, `flush()` | Beginner |
| 4 | See asynchronous publishing in a batch | Intermediate |
| 5 | Understand queue pressure and `BufferError` | Intermediate |
| 6 | Understand shutdown and lost/unobserved outcomes | Intermediate |
| 7 | Understand a few useful reliability settings | Intermediate |
| 8 | Build a small production-oriented publisher | Practical |
| 9 | Use it in an application | Practical |
| 10 | Unit-test producer behavior without a broker | Professional |
| 11 | Operational checklist and anti-patterns | Professional |
| 12 | Engineer's one-page cheat sheet | Revision |

**Boundaries:** This notebook teaches the fire-and-forget pattern. The next notebook adds **individual delivery callbacks**; advanced retry policies, the outbox pattern, consumer processing, and transactions deserve separate lessons.

---
# Section 0 — Prepare the local lab

**Objective:** Connect to a running Kafka broker and create a dedicated topic **without deleting existing topics or messages**.

### Prerequisites

- Python 3.10+ and a Jupyter kernel.
- `confluent-kafka` installed in the notebook's Python environment. If needed, run `%pip install confluent-kafka` in a separate code cell, then restart the kernel.
- A reachable local Kafka broker at `localhost:9092`, or set `KAFKA_BOOTSTRAP_SERVERS` in your environment.

**Why no `docker exec` commands?** Those depend on a particular container name, shell, and operating system. A Python Kafka producer should work whether the broker is running in Docker, locally, or remotely.

**Safe topic policy:** We generate a fresh, lab-only topic name per kernel run and create it if necessary. We **never delete topics**. A production service should not create its own topics at startup; topic lifecycle belongs to infrastructure/administration.

In [2]:
# Imports are grouped by responsibility.
import json                 # Convert Python dictionaries into JSON strings.
import logging              # Record useful operational information.
import os                   # Read environment-based configuration.
import time                 # Deadlines for bounded retries.
import uuid                 # Unique event IDs and isolated lab topic names.
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Mapping

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
TOPIC = f"tutorial_ff_{uuid.uuid4().hex[:8]}"

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("kafka.fire_and_forget.tutorial")

print("Bootstrap servers:", BOOTSTRAP)
print("Dedicated lab topic:", TOPIC)

Bootstrap servers: localhost:9092
Dedicated lab topic: tutorial_ff_30c8b58b


### 0.1 Create the topic and verify that Kafka is reachable

**Why this helper exists:** Every later lab should concentrate on producer behavior, not repeated topic setup. Kafka's `AdminClient` is used **only in this setup step**.

**Lab assumption:** Replication factor 1 is suitable for a single-broker practice environment, **not** a resilient production cluster. If you use a managed/shared cluster without create-topic permissions, ask an administrator to provision a topic and set `TOPIC` to its name.

In [3]:
def ensure_lab_topic(name: str, partitions: int = 3) -> None:
    """Create a missing lab topic; never delete or overwrite existing data."""
    admin = AdminClient({"bootstrap.servers": BOOTSTRAP})

    # The metadata request confirms the broker is reachable.
    metadata = admin.list_topics(timeout=10)
    if name in metadata.topics:
        print(f"Topic already exists: {name}")
        return

    request = NewTopic(name, num_partitions=partitions, replication_factor=1)
    admin.create_topics([request])[name].result(timeout=15)

    # Wait for the newly created topic to appear in cluster metadata.
    deadline = time.monotonic() + 10
    while name not in admin.list_topics(timeout=5).topics:
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Topic {name!r} is not visible yet")
        time.sleep(0.2)

    print(f"Created {name} with {partitions} partitions")


ensure_lab_topic(TOPIC)

Created tutorial_ff_30c8b58b with 3 partitions


---
# Section 1 — First understand the pattern

**Objective:** Be able to explain fire-and-forget without referring to code.

### The courier-office analogy

You run a website and hand **low-value analytics postcards** to a courier service:

1. **`produce()` — Submit a postcard.** The courier accepts it into an outgoing queue. This is *not* proof of arrival.
2. **Background networking — Transport the postcard.** The courier's delivery workers operate automatically. You do not manually start them by calling `poll()` or `flush()`.
3. **Broker acknowledgement — Destination acknowledges according to the configured `acks` policy.** This happens in the Kafka protocol even when your application chooses not to track delivery reports.
4. **`producer.poll(0)` — Service the courier's notification/event desk.** If you registered delivery callbacks they run here; without callbacks, you still service library events.
5. **`producer.flush(timeout)` — Before closing the office, allow outstanding work to finish.** It services events while waiting. A drained queue does **not** tell you how many messages succeeded.

### The distinction that matters

```text
Your application       Producer internals                 Kafka broker
      |                       |                               |
      |-- produce() --------->| locally queue message          |
      |<-- returns -----------|                               |
      |                       |--- background network send -->|
      |                       |<-- acknowledgement/error -----|
      |-- poll(0) ----------->| serve any ready callbacks      |
      |-- flush(timeout) ---->| service events, wait/drain     |
```

**Producer `poll()` ≠ Consumer `poll()`**. `Consumer.poll()` is for receiving topic records. We do not need a consumer to publish events.

**Definition:** *Fire-and-forget is a usage decision: do not register or act upon individual message delivery callbacks.* There is no `fire_and_forget=True` Kafka configuration.

---
# Section 2 — Publish your first real event

**Objective:** Write the smallest meaningful fire-and-forget producer.

**Why website analytics instead of orders?** A dropped page-view event is often acceptable. An unreported lost order is not.

**Event format:** A Python `dict` becomes JSON encoded as UTF-8 bytes. The `key` chooses a stable partition for related events from the same (fictional) visitor.

In [4]:
event = {
    "event_id": str(uuid.uuid4()),       # Unique event identifier.
    "event_type": "page_view",
    "visitor_id": "visitor-42",         # Synthetic ID, not real personal data.
    "page": "/products",
    "occurred_at": datetime.now(timezone.utc).isoformat(),
}

producer = Producer({"bootstrap.servers": BOOTSTRAP})

# produce() only accepts the message into the LOCAL producer queue.
producer.produce(
    topic=TOPIC,
    key=event["visitor_id"].encode("utf-8"),
    value=json.dumps(event).encode("utf-8"),
)
print("Accepted locally:", event["event_id"])

# Wait for outstanding producer work before ending this short-lived lab.
remaining = producer.flush(timeout=15)
print("Outstanding after flush:", remaining)

Accepted locally: a5903f44-85dc-4364-8413-19933aa00b1d
Outstanding after flush: 0


### What did we prove?

- `produce()` accepted the message locally (if it did not raise an exception).
- Kafka's background networking handled the actual send.
- `flush()` waited for outstanding producer work, up to 15 seconds.
- `remaining == 0` means **no outstanding producer work remained**. It does **not** prove the message was delivered successfully because we did not inspect an individual delivery report.

**Avoid this wrong statement:** “`produce() + flush()` guarantees delivery.” It does not.

**Note:** Short demos often put `flush()` immediately after one message. High-throughput applications generally **do not flush after each message**, because doing so defeats asynchronous batching.

---
# Section 3 — Understand the three methods, without extra jargon

**Objective:** Know the *responsibility* of each method and recognize what the return values actually mean.

| Method | Engineer's interpretation | What it does **not** mean |
|---|---|---|
| `produce()` | Please queue this event for asynchronous transmission | Kafka has already stored it |
| `producer.poll(0)` | Process producer events that are ready *right now* | Read messages from a topic |
| `producer.poll(0.2)` | Service events, potentially waiting up to 0.2 seconds | Force a broker acknowledgement |
| `producer.flush(10)` | Wait up to 10 seconds for outstanding producer work | Confirm that every delivery succeeded |

**Why call `poll()` even when there is no `on_delivery` callback?** It allows the client to serve pending events and keeps the producer event loop healthy. It is particularly important in high-volume producer loops. It does **not** itself send messages.

**About `len(producer)`:** It is a snapshot of outstanding producer messages and protocol requests, **not** a Kafka topic message count and **not** a delivery-success metric.

In [5]:
# A small observation experiment (outputs depend on timing and machine speed).
observation_producer = Producer({"bootstrap.servers": BOOTSTRAP})

for index in range(10):
    observation_producer.produce(TOPIC, value=f"demo-{index}".encode("utf-8"))

print("Immediately after queueing, len =", len(observation_producer))

# Process events that may already be ready; do not wait.
processed_events = observation_producer.poll(0)
print("poll(0) processed events =", processed_events)
print("After poll(0), len =", len(observation_producer))

# Wait for this lab's outstanding work before leaving the cell.
remaining = observation_producer.flush(timeout=15)
print("After flush, remaining =", remaining)

Immediately after queueing, len = 10
poll(0) processed events = 0
After poll(0), len = 10
After flush, remaining = 0


**Expected observation:** The numbers vary. A fast broker may acknowledge many events almost immediately; a slow broker may not. Do not write tests that depend on an exact `len()` value after a particular sleep.

**The durable lesson:** The producer can send and receive acknowledgements in the background; `poll()` services application-visible events; `flush()` drains outstanding work before shutdown.

---
# Section 4 — Publish a batch without blocking after every message

**Objective:** Understand the normal long-running publishing loop.

In a web application, telemetry events arrive continuously. You should not construct a new producer for every event, and you should not call `flush()` after every event. Instead, **reuse one producer**, enqueue events asynchronously, and service events regularly.

We now send 100 synthetic page views. The count of *locally accepted* messages is tracked separately from any broker-delivery claim.

In [6]:
batch_producer = Producer({"bootstrap.servers": BOOTSTRAP})
accepted_locally = 0

start = time.monotonic()

try:
    for index in range(100):
        telemetry_event = {
            "event_id": str(uuid.uuid4()),
            "event_type": "page_view",
            "visitor_id": f"visitor-{index % 10}",
            "page": "/products",
        }

        batch_producer.produce(
            topic=TOPIC,
            key=telemetry_event["visitor_id"].encode("utf-8"),
            value=json.dumps(telemetry_event).encode("utf-8"),
        )
        accepted_locally += 1

        # Service events without waiting after EACH accepted record.
        batch_producer.poll(0)
finally:
    remaining = batch_producer.flush(timeout=20)

elapsed = time.monotonic() - start
print(f"Accepted locally: {accepted_locally}")
print(f"Elapsed including final drain: {elapsed:.3f}s")
print(f"Outstanding at shutdown: {remaining}")

Accepted locally: 100
Elapsed including final drain: 0.056s
Outstanding at shutdown: 0


### Why not `flush()` inside the loop?

```python
# Avoid this in a high-throughput producer loop:
for event in events:
    producer.produce(topic, value=encode(event))
    producer.flush()  # waits after each event; blocks batching/pipelining

# Typical approach:
for event in events:
    producer.produce(topic, value=encode(event))
    producer.poll(0)  # service events, don't wait for each message
producer.flush(20)    # bounded drain when the batch/application finishes
```

**Performance rule:** Async send + occasional `poll()` enables batching and throughput. `flush()` is chiefly a **batch boundary/shutdown tool**, not the main sending mechanism.

**Production footnote:** This basic example omits local queue-full handling for clarity; Section 8 adds it.

---
# Section 5 — Queue pressure: what if the producer cannot accept more?

**Objective:** Recognize `BufferError`, understand backpressure, and avoid unbounded retry loops.

### The analogy

Your courier dispatch desk has a capacity limit. If parcels arrive faster than the company can accept/process them, the desk must say **“not now”**. That is producer **backpressure**.

With `confluent-kafka`, `produce()` can raise `BufferError` when local queue capacity is reached. Queue capacity is influenced by settings including `queue.buffering.max.messages` and `queue.buffering.max.kbytes`.

**Subtle but important:** Delivery reports that have not been served can contribute to producer queue accounting. The official librdkafka FAQ recommends polling regularly to avoid queue-full errors. However, the exact point at which a live run encounters `BufferError` depends on workload, broker/network behavior, configuration, and timing. **Never rely on an exact message count or assume a single `poll(0)` always frees space.**

### Proper response

1. If `produce()` raises `BufferError`, **that attempt was not enqueued**.
2. Call `poll(wait)` to service events and give the producer time to progress.
3. Retry the *same payload*, with a **deadline**.
4. If the deadline expires, report the failure to the caller (or hand the event to a durable fallback).

Do **not** retry broker delivery failures here: without individual delivery callbacks, fire-and-forget cannot identify those failures.

In [7]:
def enqueue_with_backpressure(
    client,
    *,
    topic: str,
    key: bytes,
    value: bytes,
    max_wait_seconds: float = 3.0,
) -> None:
    """Retry ONLY when the local Kafka queue refuses to accept a message."""
    if max_wait_seconds < 0:
        raise ValueError("max_wait_seconds cannot be negative")

    deadline = time.monotonic() + max_wait_seconds

    while True:
        try:
            # If this succeeds, the payload was queued locally.
            client.produce(topic=topic, key=key, value=value)
            return

        except BufferError:
            # No message was queued in this attempt.
            # Polling services events and allows background work to progress.
            remaining_time = deadline - time.monotonic()
            if remaining_time <= 0:
                raise TimeoutError("Local Kafka producer queue stayed full")

            client.poll(min(0.2, remaining_time))

### 5.1 Test the retry logic deterministically — no broker timing assumptions

A real Kafka queue fills at unpredictable rates. A **fake client** can deliberately reject the first two `produce()` attempts, letting us test our logic without relying on a lucky timing condition.

This is a *unit test of our own code*, not a simulation of Kafka's wire protocol.

In [8]:
class FakeQueueFullProducer:
    """Reject a configurable number of submissions, then accept one."""

    def __init__(self, reject_first: int):
        self.reject_first = reject_first
        self.attempts = 0
        self.poll_calls = 0
        self.accepted = []

    def produce(self, *, topic, key, value):
        self.attempts += 1
        if self.attempts <= self.reject_first:
            raise BufferError("Simulated local queue full")
        self.accepted.append((topic, key, value))

    def poll(self, timeout):
        self.poll_calls += 1
        return 0


fake = FakeQueueFullProducer(reject_first=2)
enqueue_with_backpressure(
    fake, topic="demo", key=b"visitor-1", value=b"{}", max_wait_seconds=3
)

assert fake.attempts == 3
assert fake.poll_calls == 2
assert len(fake.accepted) == 1
print("PASS: retry was bounded to the local-enqueue operation")

PASS: retry was bounded to the local-enqueue operation


### 5.2 Optional: Observe queue pressure with a real broker

This optional cell sets an intentionally small producer queue and attempts a burst without polling. **It may or may not raise `BufferError`** depending on broker/network timing. That variability is a feature of distributed systems, not a broken experiment.

The exercise is disabled by default because the deterministic test above already proves our application logic. If you enable it, it will clean up its producer via bounded `flush()`.

In [9]:
RUN_OPTIONAL_QUEUE_LAB = False

if RUN_OPTIONAL_QUEUE_LAB:
    stressed = Producer({
        "bootstrap.servers": BOOTSTRAP,
        "queue.buffering.max.messages": 20,
        "queue.buffering.max.kbytes": 1024,
    })
    queued = 0
    hit_full = False
    try:
        for i in range(300):
            try:
                stressed.produce(TOPIC, value=b"small-telemetry-event")
                queued += 1
            except BufferError:
                hit_full = True
                break
        print("Locally accepted before stopping:", queued)
        print("Observed BufferError:", hit_full)
        print("Producer queue snapshot:", len(stressed))
    finally:
        print("Unfinished after flush:", stressed.flush(timeout=15))
else:
    print("Optional queue-pressure lab skipped (set RUN_OPTIONAL_QUEUE_LAB=True).")

Optional queue-pressure lab skipped (set RUN_OPTIONAL_QUEUE_LAB=True).


---
# Section 6 — Shutdown and invisible delivery failures

**Objective:** Separate **finished work** from **confirmed successful delivery**.

### Why flush on shutdown?

Your application can enqueue messages much faster than they travel over the network. If the process exits abruptly, local queued messages can be lost. A bounded `flush()` during graceful shutdown gives the client time to finish outstanding work.

### What `flush()` does *not* promise

Even if `flush()` returns `0`, some messages might have failed during delivery. The producer finished processing those messages but your fire-and-forget application did not record the final per-message results.

That is the defining trade-off of this pattern.

### Two different meanings of “failure”

| Failure | Where it appears | Can basic fire-and-forget detect it? |
|---|---|---|
| The **local queue** refuses the message (`BufferError`) | Synchronous exception from `produce()` | **Yes** — handle immediately |
| Broker/network **delivery ultimately fails** after enqueueing | Final delivery outcome | **Not individually**, without a delivery callback |

**Counterexample:** A message may be accepted locally, then fail because all retries/timeouts were exhausted. `flush()` can still return zero after that failure. Don't equate “nothing outstanding” with “everything safely stored”.

### 6.1 Optional: Demonstrate a silent delivery failure

This experiment targets an intentionally unreachable broker address and uses a short delivery timeout. It provides **no per-message delivery callback**; the producer may print internal connection errors, but your application has **no individual proof of success/failure**.

> Run only in your local learning environment. This demonstrates a *bad destination*; it is not a production outage-recovery test.

In [10]:
RUN_OPTIONAL_FAILURE_LAB = False

if RUN_OPTIONAL_FAILURE_LAB:
    unreachable = Producer({
        "bootstrap.servers": "127.0.0.1:9999",  # Intentionally wrong.
        "delivery.timeout.ms": 3000,
    })

    unreachable.produce("not-a-real-topic", value=b"non-critical-test-event")
    print("Queued locally; we did not register an on_delivery callback")

    remaining = unreachable.flush(timeout=6)
    print("Outstanding after flush:", remaining)
    print("This number is NOT a count of successful deliveries")
else:
    print("Optional failure lab skipped (set RUN_OPTIONAL_FAILURE_LAB=True).")

Optional failure lab skipped (set RUN_OPTIONAL_FAILURE_LAB=True).


---
# Section 7 — Settings engineers actually need to recognize

**Objective:** Avoid memorizing 100 Kafka settings. Understand a small set and their trade-offs.

| Setting | Why it matters | Typical thought process |
|---|---|---|
| `bootstrap.servers` | How the producer discovers Kafka brokers | Environment-specific address |
| `acks` | How strong a broker acknowledgement is requested | `all` is usually preferred when reliability matters |
| `enable.idempotence` | Protects against eligible duplicate writes caused by client retries | Commonly enabled for reliable publishing |
| `delivery.timeout.ms` | Upper bound for overall delivery attempt | Choose based on operational expectations |
| `queue.buffering.max.messages` | Local queued-message count limit | Tune only when measured queue pressure warrants it |
| `linger.ms` | Small delay that may improve batching | Throughput versus latency trade-off |
| `compression.type` | Reduces wire/network size for suitable payloads | Consider for high-volume producers |

Our **production-oriented exercise** will use:

```python
{
    "bootstrap.servers": BOOTSTRAP,
    "acks": "all",
    "enable.idempotence": True,
    "delivery.timeout.ms": 30000,
    "linger.ms": 5,
}
```

**Do not confuse:** `acks=0` is a Kafka protocol choice (don't wait for broker acknowledgement). Fire-and-forget is an **application-level choice** (don't track individual outcomes). They are independent decisions.

**Single-broker lab limitation:** `acks=all` with replication factor 1 does **not** protect against losing that broker's data. Production durability also depends on replication, ISR requirements, broker/topic settings, and operations.

**For critical business events:** Prefer callback-confirmed publishing plus recovery/reconciliation, or an outbox where the event must survive a process crash. Adding `acks=all` alone does not solve end-to-end reliability.

---
# Section 8 — Build a small reusable producer (production-oriented design)

**Objective:** Encapsulate *only the fire-and-forget responsibilities*, without turning this into an 800-line framework.

### Design contract

`FireAndForgetPublisher` will:

- Accept a Python event dictionary plus an explicit partitioning key.
- Serialize to UTF-8 JSON.
- Use a **single reusable producer** instance.
- Submit **without** an individual delivery callback.
- Service producer events after enqueueing.
- Retry `BufferError` with a **deadline** rather than forever.
- Refuse new messages once shutdown begins.
- Perform a bounded flush on close and **raise** if messages remain outstanding; allow a later `close()` retry.
- Record local submission metrics, **not false delivery-success counts**.

It will **not** claim it can detect every failed broker delivery. That would contradict the pattern.

### How to read the class

Read it top-to-bottom: configuration → constructor → `send` (the normal path) → `close` (shutdown). The comments focus on **why** each responsibility exists.

In [11]:
@dataclass(frozen=True)
class FireAndForgetSettings:
    """Application settings, separate from Kafka's own property names."""

    bootstrap_servers: str
    topic: str
    max_enqueue_wait_seconds: float = 3.0
    poll_wait_seconds: float = 0.2
    shutdown_timeout_seconds: float = 35.0

    def __post_init__(self):
        if not self.bootstrap_servers or not self.topic:
            raise ValueError("bootstrap_servers and topic are required")
        if self.max_enqueue_wait_seconds < 0:
            raise ValueError("max_enqueue_wait_seconds must be >= 0")
        if self.poll_wait_seconds <= 0 or self.shutdown_timeout_seconds <= 0:
            raise ValueError("poll_wait_seconds and shutdown_timeout_seconds must be > 0")


class FireAndForgetPublisher:
    """Asynchronous publisher WITHOUT individual delivery confirmation.

    Contract:
      send() returns after LOCAL enqueue, not broker acknowledgement.
      close() tries to finish outstanding work before shutdown.

    A producer_factory can be injected in tests, so tests do not
    depend on Kafka being reachable or on unpredictable timings.
    """

    def __init__(self, settings: FireAndForgetSettings, producer_factory=Producer):
        self.settings = settings
        self._producer = producer_factory({
            "bootstrap.servers": settings.bootstrap_servers,
            "acks": "all",
            "enable.idempotence": True,
            "delivery.timeout.ms": 30000,
            "linger.ms": 5,
        })
        self._closed = False
        self._closing = False
        self.accepted_locally = 0
        self.queue_full_events = 0

    def send(self, event: Mapping[str, Any], *, key: str) -> None:
        """Accept ONE event into Kafka's local producer queue.

        Raises:
            RuntimeError: if the publisher is already closed.
            ValueError/TypeError: if the event/key is invalid.
            TimeoutError: if local queue pressure lasts too long.
            KafkaException: for other immediate Kafka client errors.
        """
        if self._closing or self._closed:
            raise RuntimeError("Cannot send: publisher is closing or closed")
        if not key:
            raise ValueError("A non-empty partition key is required")

        # Serialize once so queue-full retries resend the SAME bytes.
        payload = json.dumps(
            dict(event), separators=(",", ":"), allow_nan=False
        ).encode("utf-8")
        key_bytes = key.encode("utf-8")

        deadline = time.monotonic() + self.settings.max_enqueue_wait_seconds

        while True:
            try:
                self._producer.produce(
                    topic=self.settings.topic,
                    key=key_bytes,
                    value=payload,
                    # Deliberately NO on_delivery callback.
                )
                self.accepted_locally += 1
                break  # Accepted locally: never retry this payload now.

            except BufferError:
                self.queue_full_events += 1
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise TimeoutError(
                        "Producer did not regain local queue capacity in time"
                    )
                self._producer.poll(min(self.settings.poll_wait_seconds, remaining))

        # Poll OUTSIDE the retry try-block: a poll error must not
        # accidentally cause a second submission of an accepted event.
        self._producer.poll(0)

    def close(self) -> int:
        """Drain outstanding producer work or raise after shutdown deadline."""
        if self._closed:
            return 0  # Already completed shutdown successfully.

        # Stop accepting events; a timed-out flush may be retried.
        self._closing = True
        remaining = self._producer.flush(self.settings.shutdown_timeout_seconds)
        if remaining:
            raise TimeoutError(
                f"Shutdown incomplete: {remaining} producer messages/events remain"
            )
        self._closed = True
        # Do NOT log 'all delivered': no delivery outcomes were tracked.
        return remaining

### Why this counts as *production-oriented*, not *production complete*

The class handles local queue pressure, protects callers from accidental sends after shutdown, and surfaces a shutdown timeout. It makes **no false reliability promise**.

For an enterprise deployment, add operational concerns according to actual needs: structured metrics, tracing/correlation, schema validation, secrets/TLS/SASL configuration, lifecycle integration, concurrency safeguards, and a documented event-loss policy.

**Concurrency boundary:** This class is designed for a simple single-threaded publisher loop. The underlying producer supports concurrency, but a wrapper sharing `send()` and `close()` across threads requires synchronization of its lifecycle state. Don't interpret this notebook class as a complete cross-thread shutdown coordinator.

**Shutdown detail:** A real orchestrator should stop accepting upstream events before calling `close()`. `close()` raising on timeout must be handled by the application so a partially drained shutdown is visible.

---
# Section 9 — Use the reusable publisher in an application

**Objective:** Understand what calling code looks like; no Kafka internals need to leak into the application.

The example publishes **non-critical page-view telemetry**. It uses `try/finally` so shutdown attempts a bounded drain even if application code fails.

**Reading tip:** Compare this small cell with the full class above. Most engineers mainly remember **this usage pattern**.

In [12]:
settings = FireAndForgetSettings(
    bootstrap_servers=BOOTSTRAP,
    topic=TOPIC,
)

analytics = FireAndForgetPublisher(settings)

try:
    for index in range(20):
        event = {
            "event_id": str(uuid.uuid4()),
            "event_type": "page_view",
            "visitor_id": f"visitor-{index % 5}",
            "page": "/home" if index % 2 else "/products",
            "occurred_at": datetime.now(timezone.utc).isoformat(),
        }

        analytics.send(event, key=event["visitor_id"])

    print("Accepted locally:", analytics.accepted_locally)
    print("Queue-full events:", analytics.queue_full_events)

finally:
    try:
        analytics.close()
        print("No producer work outstanding at shutdown.")
        print("Individual delivery success is UNKNOWN (fire-and-forget).")
    except TimeoutError as error:
        logger.error("Producer shutdown incomplete: %s", error)
        raise

Accepted locally: 20
Queue-full events: 0
No producer work outstanding at shutdown.
Individual delivery success is UNKNOWN (fire-and-forget).


### Read the output carefully

- `Accepted locally: 20` means 20 calls to `produce()` succeeded locally, **not** 20 broker acknowledgements.
- `Queue-full events: 0` is common in a small local lab; it is **not** evidence that backpressure handling is unnecessary.
- `No producer work outstanding` means shutdown drained outstanding work; it does **not** mean every event succeeded.

**Safety boundary:** Don't use a fire-and-forget publisher as the sole mechanism for money movement, order creation, or inventory mutations. The next pattern adds explicit delivery outcomes.

---
# Section 10 — Unit-test our publisher WITHOUT a Kafka broker

**Objective:** Prove the wrapper's *application logic* deterministically: JSON serialization, bounded local retry, graceful drain, and rejecting new sends after close.

A fake producer implements only the `produce()`, `poll()`, and `flush()` methods used by our class. It does **not** pretend to emulate Kafka's full internals.

This is valuable because tests depending on exact broker timing are often flaky.

In [13]:
class FakeProducer:
    """Tiny fake for testing application behavior, not Kafka itself."""

    def __init__(self, config=None, *, reject_first=0, flush_remaining=0):
        self.config = config or {}
        self.reject_first = reject_first
        self.flush_remaining = flush_remaining
        self.attempts = 0
        self.records = []
        self.poll_calls = []
        self.flush_calls = []

    def produce(self, *, topic, key, value):
        self.attempts += 1
        if self.attempts <= self.reject_first:
            raise BufferError("Simulated local queue full")
        self.records.append((topic, key, value))

    def poll(self, timeout):
        self.poll_calls.append(timeout)
        return 0

    def flush(self, timeout):
        self.flush_calls.append(timeout)
        return self.flush_remaining


def fake_settings(**overrides):
    values = {"bootstrap_servers": "fake:9092", "topic": "fake-topic"}
    values.update(overrides)
    return FireAndForgetSettings(**values)

In [14]:
# TEST 1 — Proper JSON serialization + local submission.
fake = FakeProducer()
publisher = FireAndForgetPublisher(fake_settings(), producer_factory=lambda _: fake)
publisher.send({"event_type": "page_view", "count": 1}, key="visitor-1")

assert publisher.accepted_locally == 1
assert len(fake.records) == 1
assert fake.records[0][0] == "fake-topic"
assert fake.records[0][1] == b"visitor-1"
assert json.loads(fake.records[0][2]) == {"event_type": "page_view", "count": 1}
assert fake.poll_calls == [0]
assert publisher.close() == 0
print("PASS — serialization, enqueue, event service, and clean shutdown")

# TEST 2 — BufferError must retry the SAME message locally.
fake = FakeProducer(reject_first=2)
publisher = FireAndForgetPublisher(fake_settings(), producer_factory=lambda _: fake)
publisher.send({"event_type": "click"}, key="visitor-2")
assert fake.attempts == 3
assert publisher.queue_full_events == 2
assert publisher.accepted_locally == 1
assert len(fake.records) == 1
publisher.close()
print("PASS — bounded retry after temporary queue pressure")

# TEST 3 — Never retry indefinitely if the queue stays full.
fake = FakeProducer(reject_first=10**9)
publisher = FireAndForgetPublisher(
    fake_settings(max_enqueue_wait_seconds=0),
    producer_factory=lambda _: fake,
)
try:
    publisher.send({"event_type": "click"}, key="visitor-3")
    raise AssertionError("Expected local queue timeout")
except TimeoutError:
    assert publisher.accepted_locally == 0
publisher.close()
print("PASS — local queue-full retry has a deadline")

# TEST 4 — Closing with outstanding messages raises.
fake = FakeProducer(flush_remaining=2)
publisher = FireAndForgetPublisher(fake_settings(), producer_factory=lambda _: fake)
try:
    publisher.close()
    raise AssertionError("Expected incomplete shutdown timeout")
except TimeoutError:
    pass
print("PASS — incomplete drain is visible to caller")

# TEST 5 — Cannot send once shutdown has started, even after timeout.
try:
    publisher.send({"event_type": "click"}, key="visitor-4")
    raise AssertionError("Expected closed-publisher error")
except RuntimeError:
    pass

# The orchestrator may retry close() after the condition clears.
fake.flush_remaining = 0
assert publisher.close() == 0
assert publisher.close() == 0  # Idempotent after a successful drain.
print("PASS — shutdown timeout is retryable and blocks further sends")

PASS — serialization, enqueue, event service, and clean shutdown
PASS — bounded retry after temporary queue pressure
PASS — local queue-full retry has a deadline
PASS — incomplete drain is visible to caller
PASS — shutdown timeout is retryable and blocks further sends


### What these tests demonstrate — and what they don't

They demonstrate that **our wrapper**:

- Enqueues well-formed JSON bytes with a partition key.
- Retries an unaccepted message after simulated `BufferError`.
- Gives up once the enqueue deadline is reached.
- Checks the flush return value and flags incomplete shutdown.
- Refuses new messages after close (and while a previous close has timed out).

They do **not** prove Kafka delivered messages. To test real end-to-end delivery, use a separate integration test with a real broker and verify through a consumer (or migrate to callback-confirmed publishing when acknowledgement tracking matters).

---
# Section 11 — Production decisions and common mistakes

**Objective:** Know which decisions are required before using this pattern in a real service.

| Mistake | Why it's a problem | Better approach |
|---|---|---|
| Treating successful `produce()` as delivery | It only means local enqueue | Distinguish **accepted locally** from **acknowledged** |
| Saying `flush()==0` proves success | Failures can also finish | Track delivery callbacks if success matters |
| Flushing after every event | Blocks batching and throughput | `poll(0)` during sending, bounded `flush()` at shutdown |
| Never servicing producer events | Events may accumulate, queue pressure worsens | Poll regularly |
| Infinite retry on `BufferError` | Can hang during broker outages | Deadline + backpressure/fallback policy |
| Creating producers per request | Extra resources/connections | Reuse a process-scoped producer |
| Publishing financial events fire-and-forget | Losses are not individually observable | Use confirmed publishing and durable recovery |
| Destroying/recreating topics in application code | Data loss / wrong privileges | Provision topics outside the producer service |
| Using `acks=all` on a one-broker lab as proof of durability | No replica survives broker loss | Production replication + ISR policy |

### Pre-production checklist

- [ ] Is **occasional event loss acceptable** for this event type?
- [ ] Are Kafka bootstrap addresses, TLS/SASL credentials, and timeouts environment-based?
- [ ] Does the producer reuse one long-lived client rather than constructing one per message?
- [ ] Are keys and payload schemas deliberately chosen and versioned?
- [ ] Is local queue pressure handled with a **bounded** policy?
- [ ] Are lifecycle and shutdown events integrated with the service manager?
- [ ] Are queue depth, publish rate, `BufferError`, and flush timeouts monitored?
- [ ] Is the absence of per-message delivery confirmation documented?
- [ ] Is there an explicit response to data loss, process crashes, and restarts?

### When to change patterns

If the requirement is **“Tell me whether event X reached Kafka, and act if it didn't”**, then **this is the wrong pattern**. Use the callback-confirmed producer next. If the requirement is **“An accepted business transaction must never be lost, even if the process crashes”**, then callbacks alone may be insufficient; study the transactional outbox or another durable recovery mechanism.

---
# Section 12 — What engineers actually remember

**Fire-and-forget** = queue/send asynchronously **without tracking each message's final delivery outcome**.

### The only three methods

| Method | Memory shortcut |
|---|---|
| `produce()` | **QUEUE** — accepted locally, not confirmed delivered |
| `poll(0)` | **SERVICE** — process available producer events |
| `flush(timeout)` | **DRAIN** — wait for pending work, especially on shutdown |

**The actual sending** is performed by Kafka's background networking.

### Normal usage pattern

```python
producer = Producer({"bootstrap.servers": BOOTSTRAP})
try:
    for event in events:
        producer.produce(TOPIC, value=serialize(event))  # no delivery callback
        producer.poll(0)                                  # service events
finally:
    remaining = producer.flush(timeout=30)                # bounded shutdown
    if remaining:
        raise TimeoutError("Producer did not fully drain")
```

This is a **memory aid**, not a copy/paste production service: the real class in Section 8 also handles `BufferError` with a deadline.

### Remember these five engineering facts

1. `produce()` succeeding is **not** proof Kafka received the event.
2. `poll()` services producer events; **it does not send messages or read topics**.
3. `flush()` drains outstanding work; **zero remaining is not proof all succeeded**.
4. `BufferError` means the **local producer queue** refused this attempt; apply bounded backpressure.
5. Use fire-and-forget only when **lack of individual delivery visibility is acceptable**.

### Mini-checkpoint — answer without scrolling

1. If I call `produce()` and exit immediately, what can go wrong?
2. Who *actually* performs Kafka network sends?
3. Does `producer.poll(0)` read Kafka topic messages?
4. How is a `BufferError` different from a final broker delivery failure?
5. Why doesn't `flush()==0` prove that 100/100 events succeeded?
6. When would you choose callback-confirmed publishing instead?

> **Next notebook:** Callback-Confirmed Producer — keep the async workflow, add a delivery callback and a policy for individual delivery failures.

---
## Reference material (optional — engineers look this up)

- [Confluent Python client API: `Producer.produce()`, `poll()`, `flush()`, `len()`](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html)
- [Confluent Python client overview: asynchronous writes](https://docs.confluent.io/kafka-clients/python/current/overview.html)
- [Official Confluent Python Producer example](https://github.com/confluentinc/confluent-kafka-python/blob/master/examples/producer.py)
- [librdkafka FAQ: queue limits and servicing delivery reports](https://github.com/confluentinc/librdkafka/wiki/FAQ)

**Version note:** Kafka client defaults/configuration and exact event accounting can change. For production decisions, verify behavior against your deployed `confluent-kafka` and `librdkafka` versions rather than relying on a notebook's timing-specific observations.